# Phase 1: 도메인 타입 + 엔지니어 데이터 모델

PROCESS.md 기반 구조화된 타입 확인 + 샘플 데이터 이중 텍스트(capability/experience) 검증

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [2]:
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES

print(f"총 {len(SAMPLE_ENGINEER_PROFILES)}명의 엔지니어 프로필 로드")
for p in SAMPLE_ENGINEER_PROFILES:
    print(f"  {p.engineer_id:8s} | {p.grade:15s} | {p.engineer_role:10s} | {p.employment_type:10s}")

총 9명의 엔지니어 프로필 로드
  eng-001  | SENIOR          | DEVELOPER  | FULL_TIME 
  eng-002  | INTERMEDIATE    | DEVELOPER  | FULL_TIME 
  eng-003  | SENIOR          | DEVELOPER  | FULL_TIME 
  eng-004  | INTERMEDIATE    | DEVELOPER  | FREELANCER
  eng-005  | INTERMEDIATE    | DEVELOPER  | FULL_TIME 
  pub-001  | INTERMEDIATE    | PUBLISHER  | FULL_TIME 
  qa-001   | SENIOR          | QA         | FULL_TIME 
  des-001  | SENIOR          | DESIGNER   | FREELANCER
  pln-001  | SENIOR          | PLANNER    | FULL_TIME 


In [3]:
# capability_text vs experience_text 비교
profile = SAMPLE_ENGINEER_PROFILES[0]
print(f"=== {profile.engineer_id} ===")
print(f"\n[capability_text]\n{profile.capability_text}")
print(f"\n[experience_text]\n{profile.experience_text}")

=== eng-001 ===

[capability_text]
Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Docker

정보처리기사

[experience_text]
[소개]
MSA 아키텍처 기반 백엔드 전문. 제조업/물류 도메인 경험 다수.

[프로젝트 경험]
현대모비스(2023~2025): ERP 재고관리 모듈 개발. Spring Batch로 야간 정산 처리 자동화. 포지션: 백엔드 리드.
LG CNS(2021~2023): 제조업 MES 시스템 API 개발. Oracle DB 쿼리 최적화로 조회속도 40% 개선. 포지션: 백엔드 개발자.
삼성SDS(2017~2021): 물류 ERP 시스템 개발. Spring Boot 기반 REST API 설계. 포지션: 백엔드 개발자.

[경력]
현대오토에버(2015~2017): 백엔드 개발자. ERP 연동 API 개발.


In [4]:
from embedding_retrieval.types import ClientRequest, PositionRequest

# PROCESS.md 예시 클라이언트 요청 구성
request = ClientRequest(
    project_name="현대차 ERP 시스템 개발",
    description="제조업 ERP 시스템 재구축 프로젝트",
    weights={"capability": 0.6, "experience": 0.4},
    positions=[
        PositionRequest(
            position="PL",
            engineer_role="DEVELOPER",
            grades=["SENIOR", "INTERMEDIATE"],
            skills=["Java", "Spring"],
            engineer_cnt=1,
            etc="팀장 역할 필수",
        ),
        PositionRequest(
            position="백엔드 개발자",
            engineer_role="DEVELOPER",
            grades=["SENIOR", "INTERMEDIATE"],
            skills=["Java", "Spring"],
            engineer_cnt=3,
            etc="현대차 프로젝트 경험 우대",
        ),
        PositionRequest(
            position="프론트엔드 개발자",
            engineer_role="DEVELOPER",
            grades=["SENIOR", "INTERMEDIATE"],
            skills=["React"],
            engineer_cnt=2,
            etc="차트.js 경험 우대",
        ),
    ],
)

print(f"프로젝트: {request.project_name}")
print(f"포지션 수: {len(request.positions)}")
for pos in request.positions:
    print(f"  {pos.position} | {pos.engineer_role} | {pos.grades} | {pos.skills} | {pos.engineer_cnt}명")

프로젝트: 현대차 ERP 시스템 개발
포지션 수: 3
  PL | DEVELOPER | ['SENIOR', 'INTERMEDIATE'] | ['Java', 'Spring'] | 1명
  백엔드 개발자 | DEVELOPER | ['SENIOR', 'INTERMEDIATE'] | ['Java', 'Spring'] | 3명
  프론트엔드 개발자 | DEVELOPER | ['SENIOR', 'INTERMEDIATE'] | ['React'] | 2명


In [5]:
from embedding_retrieval.weights import get_weights

# 포지션별 가중치 확인
test_cases = [
    ("PL", "DEVELOPER"),
    ("백엔드 개발자", "DEVELOPER"),
    ("프론트엔드 개발자", "DEVELOPER"),
    ("PM", "PLANNER"),
    ("QA 리드", "QA"),
    ("디자이너", "DESIGNER"),
]

print(f"{'position':20s} {'role':12s} {'cap_w':6s} {'exp_w':6s}")
print("-" * 48)
for pos, role in test_cases:
    cap_w, exp_w = get_weights(pos, role)
    print(f"{pos:20s} {role:12s} {cap_w:.1f}   {exp_w:.1f}")

position             role         cap_w  exp_w 
------------------------------------------------
PL                   DEVELOPER    0.2   0.8
백엔드 개발자              DEVELOPER    0.7   0.3
프론트엔드 개발자            DEVELOPER    0.7   0.3
PM                   PLANNER      0.2   0.8
QA 리드                QA           0.5   0.5
디자이너                 DESIGNER     0.4   0.6


## 실제 임베딩 테스트 capability_text / experience_text 를 Google Embedding 모델로 임베딩하고 코사인 유사도를 비교한다.

In [4]:
import numpy as np
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores.upstash import UpstashVectorStoreAdapter
from embedding_retrieval.compat import Document

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)
print(f"Embedding provider: {config.embedding_provider} / {config.embedding_model}")

url = config.vector_store_kwargs["url"]
token = config.vector_store_kwargs["token"]

# Upstash 네임스페이스 분리: capability / experience
cap_store = UpstashVectorStoreAdapter(embeddings=embeddings, url=url, token=token, namespace="capability")
exp_store = UpstashVectorStoreAdapter(embeddings=embeddings, url=url, token=token, namespace="experience")

# Document 변환 + Upstash 업로드
cap_docs = []
exp_docs = []
for p in SAMPLE_ENGINEER_PROFILES:
    meta = {
        "engineer_id": p.engineer_id,
        "grade": p.grade,
        "status": p.status,
        "engineer_role": p.engineer_role,
        "employment_type": p.employment_type,
    }
    cap_docs.append(Document(page_content=p.capability_text, metadata=meta))
    exp_docs.append(Document(page_content=p.experience_text, metadata=meta))

cap_store.add_documents(cap_docs)
exp_store.add_documents(exp_docs)
print(f"Upstash 업로드 완료: capability {len(cap_docs)}건, experience {len(exp_docs)}건")

Embedding provider: google_genai / gemini-embedding-2-preview
Upstash 업로드 완료: capability 9건, experience 9건


In [5]:
# Upstash에서 실제 벡터 검색
# capability 검색: "Java / Spring" → 백엔드 엔지니어가 상위여야 함
print("=== capability 검색 ===")
cap_results = cap_store.similarity_search("Java / Spring", top_k=5)
for r in cap_results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  | {r.document.page_content[:60].strip()}")

print()

# experience 검색: "제조업 ERP 시스템 개발 경험" → 제조업 도메인 경험자가 상위여야 함
print("=== experience 검색: '제조업 ERP 시스템 개발 경험' ===")
exp_results = exp_store.similarity_search("제조업 ERP 시스템 개발 경험", top_k=5)
for r in exp_results:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  | {r.document.page_content[:60].strip()}")

=== capability 검색 ===
  eng-002   score: 0.9576  | Java / Spring Boot / Spring Security / MySQL / Kafka / AWS
  eng-001   score: 0.9091  | Java / Spring Boot / Spring Batch / PostgreSQL / Redis / Doc
  qa-001    score: 0.8911  | Selenium / Appium / JMeter / Jira / Confluence / TestRail
  eng-005   score: 0.8876  | Python / FastAPI / MongoDB / Docker / Kubernetes / Celery
  eng-004   score: 0.8729  | React / JavaScript / Vue.js / Chart.js / CSS / Webpack

=== experience 검색: '제조업 ERP 시스템 개발 경험' ===
  eng-001   score: 0.8544  | [소개]
MSA 아키텍처 기반 백엔드 전문. 제조업/물류 도메인 경험 다수.

[프로젝트 경험]
현대모비스(
  eng-002   score: 0.8309  | [소개]
자동차/반도체 제조 도메인 백엔드 개발. 이벤트 기반 아키텍처 경험.

[프로젝트 경험]
현대차(2
  eng-005   score: 0.8110  | [소개]
물류/이커머스 도메인 백엔드 개발. MSA 및 컨테이너 오케스트레이션 경험.

[프로젝트 경험]
쿠
  eng-004   score: 0.8100  | [소개]
제조/광고 도메인 프론트엔드 개발. 차트 기반 대시보드 경험.

[프로젝트 경험]
포스코(2024~
  eng-003   score: 0.8017  | [소개]
자동차/금융 도메인 프론트엔드 전문. 대시보드 및 데이터 시각화 경험 풍부.

[프로젝트 경험]
기


In [ ]:
# 스칼라 필터 + 벡터 검색 조합 테스트
# DEVELOPER만 필터 + capability 검색
print("=== capability 검색 + 필터: engineer_role='DEVELOPER' ===")
filtered = cap_store.similarity_search("Java / Spring", top_k=5, filter="engineer_role = 'DEVELOPER'")
for r in filtered:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  role: {r.metadata['engineer_role']}")

print()

# SENIOR만 필터 + experience 검색
print("=== experience 검색 + 필터: grade='SENIOR' ===")
filtered = exp_store.similarity_search("제조업 ERP 시스템 개발 경험", top_k=5, filter="grade = 'SENIOR'")
for r in filtered:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}  grade: {r.metadata['grade']}")